In [1]:
# GPU setup with explicit library loading
import os, sys, ctypes
os.environ['JAX_PLATFORM_NAME'] = 'gpu'

# Try loading cuSPARSE directly with full path
cusparse_lib = '/mnt/weka/home/lzchen/miniconda3/envs/ferminet/targets/x86_64-linux/lib/libcusparse.so.12.5.10.65'
ctypes.cdll.LoadLibrary(cusparse_lib)

import time, numpy as np, jax, jax.numpy as jnp, matplotlib.pyplot as plt
from jax import random
print("Backend:", jax.default_backend(), "| Devices:", len(jax.devices()))


Backend: gpu | Devices: 8


In [ ]:
d = 64; dk = 64; dtype = jnp.float32
key = random.PRNGKey(0)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

# ----------------- Utils ------------------
def glorot_uniform(k, shape, dtype=dtype):
    fan_in, fan_out = shape[-2], shape[-1]
    lim = np.sqrt(6.0 / (fan_in + fan_out))
    return random.uniform(k, shape, minval=-lim, maxval=lim, dtype=dtype)

def init_params(k, d=64, dk=64):
    k, kq, kk, kv, ko = random.split(k, 5)
    return dict(
        WQ=glorot_uniform(kq, (d, dk)),
        WK=glorot_uniform(kk, (d, dk)),
        WV=glorot_uniform(kv, (d, d)),
        WO=glorot_uniform(ko, (d, d)),
    )

def softmax_rows(x):
    x = x - jnp.max(x, axis=-1, keepdims=True)
    ex = jnp.exp(x)
    return ex / jnp.sum(ex, axis=-1, keepdims=True)

# --------- Forward (single attention layer) ----------
def attn_forward(X, P):
    WQ, WK, WV, WO = P["WQ"], P["WK"], P["WV"], P["WO"]
    dk = WQ.shape[-1]
    Q = X @ WQ
    K = X @ WK
    V = X @ WV
    S = (Q @ K.T) / jnp.sqrt(dk)
    A = softmax_rows(S)
    Y = (A @ V) @ WO
    return Y, (Q, K, V, A, dk, WQ, WK, WV, WO)

# Black-box forward (autodiff will do JVPs)
attn_blackbox_jit = jax.jit(lambda X, P: attn_forward(X, P)[0])

# --------- Diagonal trace via JVP loop ---------------
def divergence_by_diag_jvp(f, X):
    N, d_ = X.shape
    def diag_entry(i, j):
        Eij = jnp.zeros_like(X).at[i, j].set(1.0)
        _, dF = jax.jvp(f, (X,), (Eij,))
        return dF[i, j]
    vals = jax.vmap(lambda i: jax.vmap(lambda j: diag_entry(i, j))(jnp.arange(d_)))(jnp.arange(N))
    return jnp.sum(vals)

# Mark function arg as static for jit
divergence_by_diag_jvp_jit = jax.jit(divergence_by_diag_jvp, static_argnums=(0,))

# --------- Closed-form exact divergence --------------
def divergence_attn_closed_form(X, P):
    Y, (Q, K, V, A, dk, WQ, WK, WV, WO) = attn_forward(X, P)
    V_eff = V @ WO
    Y_eff = Y
    M    = K @ WQ.T      # (N, d)
    Nmat = Q @ WK.T      # (N, d)

    t0 = jnp.trace(A) * jnp.trace(WV @ WO)
    c  = jnp.sum(A, axis=0)                       # (N,)
    t1 = jnp.dot(c, jnp.sum(V_eff * M, axis=1))   # c^T rowdot(V_eff, M)
    AM = A @ M
    t2 = jnp.sum(AM * Y_eff)                      # <AM, Y>_F
    diagA = jnp.diag(A)
    diff  = V_eff - Y_eff
    t3 = jnp.dot(diagA, jnp.sum(Nmat * diff, axis=1))
    return t0 + (t1 - t2 + t3) / jnp.sqrt(dk)

divergence_attn_closed_form_jit = jax.jit(divergence_attn_closed_form)

# ----------------- Timing helpers --------------------
def time_once(fun, *args, warmup=True, repeat=3, label=""):
    if warmup:
        out = fun(*args)
        (out if hasattr(out, "block_until_ready") else jnp.asarray(out)).block_until_ready()
    times = []
    for _ in range(repeat):
        t0 = time.time()
        out = fun(*args)
        (out if hasattr(out, "block_until_ready") else jnp.asarray(out)).block_until_ready()
        t1 = time.time()
        times.append(t1 - t0)
    tmin = min(times)
    print(f"{label:28s} {tmin*1e3:8.2f} ms")
    return tmin

In [ ]:
Ps = init_params(key, d=d, dk=dk)
Ns_small = [16, 32, 48, 64, 80, 96, 128]        # compare both ways
Ns_large = [128, 192, 256, 384, 512, 768, 1024] # closed-form scaling

rows = []

print("\n== Small Ns: compare diag-loop vs closed-form ==")
for N in Ns_small:
    kx = random.fold_in(key, N)
    X = random.normal(kx, (N, d), dtype=dtype)

    # 1) Diag-loop + standard autodiff
    f_blackbox = lambda Z: attn_blackbox_jit(Z, Ps)
    t1 = time_once(lambda: divergence_by_diag_jvp_jit(f_blackbox, X),
                    label=f"N={N:4d}  diag+autodiff")

    # 2) Closed-form exact (no loop)
    t2 = time_once(lambda: divergence_attn_closed_form_jit(X, Ps),
                    label=f"N={N:4d}  closed-form")

    # --- correctness: print + assert they match ---
    div_diag  = divergence_by_diag_jvp_jit(f_blackbox, X)
    div_close = divergence_attn_closed_form_jit(X, Ps)
    diff = float(jnp.max(jnp.abs(div_diag - div_close)))
    print(f"  match?  |div_diag - div_closed| = {diff:.3e}")
    assert jnp.allclose(div_diag, div_close, rtol=1e-5, atol=1e-6)

    rows.append((N, t1, t2))

print("\n== Large Ns: closed-form only ==")
for N in Ns_large:
    kx = random.fold_in(key, N)
    X = random.normal(kx, (N, d), dtype=dtype)
    t2 = time_once(lambda: divergence_attn_closed_form_jit(X, Ps),
                    label=f"N={N:4d}  closed-form")
    rows.append((N, np.nan, t2))

# -------- Table --------
print("\nSummary (min ms over 3 runs):")
print(f"{'N':>6}  {'diag+autodiff':>15}  {'closed-form':>12}")
for N, t1, t2 in rows:
    s1 = f"{t1*1e3:8.1f}" if np.isfinite(t1) else "   n/a  "
    s2 = f"{t2*1e3:8.1f}"
    print(f"{N:6d}  {s1:>15}  {s2:>12}")

# -------- Plot (log time) --------
Ns_plot = [r[0] for r in rows]
y1 = [r[1]*1e3 if np.isfinite(r[1]) else np.nan for r in rows]
y2 = [r[2]*1e3 for r in rows]
plt.figure()
plt.plot(Ns_plot, y1, marker="o", label="diag+autodiff (O(N^3 d))")
plt.plot(Ns_plot, y2, marker="o", label="closed-form (O(N^2 d))")
plt.xlabel("Sequence length N")
plt.ylabel("Time (ms, min of 3) [log]")
plt.yscale("log")              # << log-time visualization
plt.title("Self-attention divergence: runtime vs N (d=64)")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# pip install --upgrade "jax[cpu]"
import jax
import jax.numpy as jnp


In [ ]:
# pip install "jax[cpu]" -q
import jax
import jax.numpy as jnp

# -----------------------------
# Problem setup
# -----------------------------
N, d, dv = 3, 2, 2
key = jax.random.PRNGKey(0)

def split_qkv(x):
    Q = x[:N*d].reshape(N, d)
    K = x[N*d:2*N*d].reshape(N, d)
    V = x[2*N*d:].reshape(N, dv)
    return Q, K, V

D = N*d*2 + N*dv

# -----------------------------
# Attention core
# -----------------------------
def attention(Q, K, V):
    S = (Q @ K.T) / jnp.sqrt(d)
    A = jax.nn.softmax(S, axis=1)
    return A @ V

# -----------------------------
# Analytic JVP and double-JVP
# -----------------------------
def attention_jvp(Q, K, V, dQ, dK, dV):
    S = (Q @ K.T) / jnp.sqrt(d)
    A = jax.nn.softmax(S, axis=1)
    dS = (dQ @ K.T + Q @ dK.T) / jnp.sqrt(d)
    r = jnp.sum(A * dS, axis=1, keepdims=True)
    dA = A * (dS - r)
    dY = dA @ V + A @ dV
    return dY

def attention_d2(Q, K, V, dQ, dK, dV):
    S = (Q @ K.T) / jnp.sqrt(d)
    A = jax.nn.softmax(S, axis=1)
    dS  = (dQ @ K.T + Q @ dK.T) / jnp.sqrt(d)
    d2S = (2 * (dQ @ dK.T)) / jnp.sqrt(d)

    r = jnp.sum(A * dS, axis=1, keepdims=True)
    phi = jnp.sum(A * (dS * dS), axis=1, keepdims=True)
    e = jnp.sum(A * d2S, axis=1, keepdims=True)

    dA = A * (dS - r)
    d2A = A * (d2S + dS * dS - 2 * r * dS + (2 * r * r - phi - e))
    d2Y = d2A @ V + 2 * (dA @ dV)
    return d2Y

# -----------------------------
# Scalar probe for Laplacian test
# -----------------------------
P = jax.random.normal(key, (N, dv))

def g(x):
    Q, K, V = split_qkv(x)
    Y = attention(Q, K, V)
    return jnp.vdot(P, Y).real

# -----------------------------
# Compute Laplacian via autodiff
# -----------------------------
def laplacian_autodiff(x):
    H = jax.hessian(g)(x)
    return jnp.trace(H)

# -----------------------------
# Compute Laplacian via analytic formula
# -----------------------------
def laplacian_formula(x):
    Q, K, V = split_qkv(x)
    I = jnp.eye(x.size)
    q_idx = jnp.arange(0, N*d)
    k_idx = jnp.arange(N*d, 2*N*d)
    v_idx = jnp.arange(2*N*d, 2*N*d + N*dv)

    def dir_to_qkv(v):
        dQ = v[q_idx].reshape(N, d)
        dK = v[k_idx].reshape(N, d)
        dV = v[v_idx].reshape(N, dv)
        return dQ, dK, dV

    def per_dir(v):
        dQ, dK, dV = dir_to_qkv(v)
        d2Y = attention_d2(Q, K, V, dQ, dK, dV)
        return jnp.vdot(P, d2Y)

    return jnp.sum(jax.vmap(per_dir)(I))

# -----------------------------
# Verification
# -----------------------------
x0 = jax.random.normal(jax.random.PRNGKey(42), (D,))
lap_autodiff = laplacian_autodiff(x0)
lap_formula = laplacian_formula(x0)

print("Laplacian via autodiff :", lap_autodiff)
print("Laplacian via formula  :", lap_formula)
print("Close?                 :", jnp.allclose(lap_autodiff, lap_formula, atol=1e-5))


In [8]:
# Make sure JAX uses GPU (CUDA)
# pip install -U "jax[cuda12_pip]" jaxlib==0.4.30 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
import jax
import jax.numpy as jnp
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

print("JAX backend:", jax.default_backend())

# -----------------------------
# Core functions (same as before)
# -----------------------------
def attention(Q, K, V):
    S = (Q @ K.T) / jnp.sqrt(Q.shape[-1])
    A = jax.nn.softmax(S, axis=1)
    return A @ V

def attention_d2(Q, K, V, dQ, dK, dV):
    d = Q.shape[-1]
    S = (Q @ K.T) / jnp.sqrt(d)
    A = jax.nn.softmax(S, axis=1)

    dS  = (dQ @ K.T + Q @ dK.T) / jnp.sqrt(d)
    d2S = (2 * (dQ @ dK.T)) / jnp.sqrt(d)

    r = jnp.sum(A * dS, axis=1, keepdims=True)
    phi = jnp.sum(A * (dS * dS), axis=1, keepdims=True)
    e = jnp.sum(A * d2S, axis=1, keepdims=True)

    dA = A * (dS - r)
    d2A = A * (d2S + dS * dS - 2 * r * dS + (2 * r * r - phi - e))
    d2Y = d2A @ V + 2 * (dA @ dV)
    return d2Y

def split_qkv(x, N, d, dv):
    Q = x[:N*d].reshape(N, d)
    K = x[N*d:2*N*d].reshape(N, d)
    V = x[2*N*d:].reshape(N, dv)
    return Q, K, V

def laplacian_formula(x, N, d, dv, P):
    Q, K, V = split_qkv(x, N, d, dv)
    I = jnp.eye(x.size)
    q_idx = jnp.arange(0, N*d)
    k_idx = jnp.arange(N*d, 2*N*d)
    v_idx = jnp.arange(2*N*d, 2*N*d + N*dv)

    def dir_to_qkv(v):
        dQ = v[q_idx].reshape(N, d)
        dK = v[k_idx].reshape(N, d)
        dV = v[v_idx].reshape(N, dv)
        return dQ, dK, dV

    def per_dir(v):
        dQ, dK, dV = dir_to_qkv(v)
        d2Y = attention_d2(Q, K, V, dQ, dK, dV)
        return jnp.vdot(P, d2Y)

    return jnp.sum(jax.vmap(per_dir)(I))

# -----------------------------
# Autodiff Laplacian
# -----------------------------
def g(x, N, d, dv, P):
    Q, K, V = split_qkv(x, N, d, dv)
    return jnp.vdot(attention(Q, K, V), P)

def laplacian_autodiff(x, N, d, dv, P):
    H = jax.hessian(lambda y: g(y, N, d, dv, P))(x)
    return jnp.trace(H)

# -----------------------------
# Timing experiment
# -----------------------------
def scaling_test():
    dims = [(8, 8, 16), (16, 8, 16), (32, 8, 16), (48, 8, 16), (64, 8, 16), (80, 8, 16), (96, 8, 16), (128, 8, 16), (192, 8, 16), (256, 8, 16), (384, 8, 16), (512, 8, 16), (768, 8, 16), (1024, 8, 16)]
    times_formula, times_auto = [], []
    match = []

    for N, d, dv in tqdm(dims, desc="Testing sequence lengths"):
        D = N*d*2 + N*dv
        x = jax.random.normal(jax.random.PRNGKey(0), (D,))
        P = jax.random.normal(jax.random.PRNGKey(1), (N, dv))

        # warm-up compile
        laplacian_formula(x, N, d, dv, P).block_until_ready()
        laplacian_autodiff(x, N, d, dv, P).block_until_ready()

        # measure analytic
        t0 = time.time()
        lap_f = laplacian_formula(x, N, d, dv, P).block_until_ready()
        t1 = time.time()

        # measure autodiff
        t2 = time.time()
        lap_a = laplacian_autodiff(x, N, d, dv, P).block_until_ready()
        t3 = time.time()

        times_formula.append(t1 - t0)
        times_auto.append(t3 - t2)
        match.append(jnp.allclose(lap_f, lap_a, atol=1e-5))

        print(f"N={N:3d}: Lap(form)={lap_f:.6f}, Lap(auto)={lap_a:.6f}, match={match[-1]}")
        
    Ns = [n for n, _, _ in dims]
    plt.figure(figsize=(7,5))
    plt.plot(Ns, np.sqrt(times_formula), "o-", label="√(Analytic time)")
    plt.plot(Ns, np.sqrt(times_auto), "s--", label="√(Autodiff time)")
    plt.xlabel("Sequence length N")
    plt.ylabel("√(Runtime) (s½)")
    plt.title("Scaling check: √time vs N  (Analytic ~ N², Auto ~ N³)")
    plt.grid(True)
    plt.legend()
    plt.show()

    for n, tf, ta in zip(Ns, times_formula, times_auto):
        print(f"N={n:3d}: analytic={tf:.3f}s, autodiff={ta:.3f}s, ratio={ta/tf:.1f}x")

scaling_test()


JAX backend: gpu


Testing sequence lengths:   7%|▋         | 1/14 [00:00<00:01,  9.73it/s]

N=  8: Lap(form)=-2.749237, Lap(auto)=-2.748910, match=False


Testing sequence lengths:  14%|█▍        | 2/14 [00:00<00:01,  9.87it/s]

N= 16: Lap(form)=-1.171961, Lap(auto)=-1.172278, match=False
N= 32: Lap(form)=-2.029868, Lap(auto)=-2.026777, match=False


...

N=128: Lap(form)=-3.608954, Lap(auto)=-3.607834, match=False
N=192: Lap(form)=-5.633762, Lap(auto)=-5.635782, match=False


Testing sequence lengths:  71%|███████▏  | 10/14 [00:01<00:00,  8.38it/s]

N=256: Lap(form)=-5.095284, Lap(auto)=-5.094999, match=False


W1009 07:42:12.598619 3234395 bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 6.75GiB (rounded to 7247757312)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W1009 07:42:12.598781 3234395 bfc_allocator.cc:512] ***************__________*********************************************************************______
E1009 07:42:12.598796 3234395 pjrt_stream_executor_client.cc:3314] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 7247757312 bytes. [tf-allocator-allocation-error='']
Testing sequence lengths:  71%|███████▏  | 10/14 [00:21<00:08,  2.12s/it]


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 7247757312 bytes.